# SpaceX Falcon 9 - Interactive Visual Analytics with Folium

Build an interactive map of all Falcon 9 launch sites, color-coded by landing outcome, with a MarkerCluster layer and distance lines to nearby infrastructure.

In [1]:
import folium
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
import pandas as pd
import math

df = pd.read_csv("dataset_part_1.csv")
landing_outcomes = df['Outcome'].value_counts()
bad_outcomes = set(landing_outcomes.keys()[[1,3,5,6,7]])
df['Class'] = df['Outcome'].apply(lambda o: 0 if o in bad_outcomes else 1)
df.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


## Build a dataframe of unique launch sites with their coordinates

In [2]:
launch_sites_df = df.groupby('LaunchSite', as_index=False).agg(
    Lat=('Latitude', 'first'),
    Long=('Longitude', 'first')
)
launch_sites_df

,LaunchSite,Lat,Long
0,CCAFS SLC 40,28.561857,-80.577366
1,KSC LC 39A,28.608058,-80.603956
2,VAFB SLC 4E,34.632093,-120.610829


## Mark every launch site on the map

In [3]:
site_map = folium.Map(location=[28.57, -80.65], zoom_start=4.5)

for _, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    circle = folium.Circle(coordinate, radius=1000, color='#0B3D91', fill=True).add_child(folium.Popup(row['LaunchSite']))
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
                     html='<div style="font-size: 12px; color:#0B3D91;"><b>%s</b></div>' % row['LaunchSite'])
    )
    site_map.add_child(circle)
    site_map.add_child(marker)

site_map

## Mark every individual launch, color-coded green (success) / red (failure), grouped with MarkerCluster

In [4]:
marker_cluster = MarkerCluster()
df['marker_color'] = df['Class'].map({1: 'green', 0: 'red'})
site_map.add_child(marker_cluster)

for _, launch in df.iterrows():
    marker = folium.Marker(
        [launch['Latitude'], launch['Longitude']],
        icon=folium.Icon(color='white', icon_color=launch['marker_color']),
        popup=f"{launch['LaunchSite']} - {'Success' if launch['Class']==1 else 'Failure'}"
    )
    marker_cluster.add_child(marker)

site_map

## Add a MousePosition widget to read coordinates while exploring the map

In [5]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright', separator=' Long: ', empty_string='NaN',
    lng_first=False, num_digits=20, prefix='Lat:',
    lat_formatter=formatter, lng_formatter=formatter,
)
site_map.add_child(mouse_position)

## Draw a distance line from a launch site to the nearest coastline

In [6]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c_ = 2 * math.asin(math.sqrt(a))
    return R * c_

# Nearest coastline point to CCAFS SLC 40, read visually from the map / MousePosition widget
coastline_lat, coastline_lon = 28.56367, -80.56784
launch_site_lat, launch_site_lon = 28.563197, -80.576820
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
                 html='<div style="font-size: 12px; color:#FC3D21;"><b>%.2f KM</b></div>' % distance_coastline)
)
lines = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]], weight=1)
site_map.add_child(distance_marker)
site_map.add_child(lines)
print(f"Distance from CCAFS SLC 40 to nearest coastline point: {distance_coastline:.2f} km")

Distance from CCAFS SLC 40 to nearest coastline point: 0.88 km


## Save the interactive map to a standalone HTML file

In [7]:
site_map.save("spacex_launch_sites_map.html")
print("Saved spacex_launch_sites_map.html")

Saved spacex_launch_sites_map.html
